<a href="https://colab.research.google.com/github/Ashwini9713/Digital-forensics-/blob/main/exp15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
"""
Experiment 1.2 — Malware Hash Analysis
========================================
Objective: Learn malware detection using cryptographic hashes.

This tool:
  1. Generates MD5 / SHA-1 / SHA-256 hashes for a set of sample files
  2. Queries the VirusTotal public API (v3) for known-malware signatures
  3. Builds a local hash-based detection system (CSV database + lookups)
  4. Analyzes hash collision risk (birthday-bound math, not exploit generation)
  5. Demonstrates hash-family clustering with a simplified fuzzy hash

No actual malware is created, downloaded, or executed. Sample "files" used
here are harmless synthetic text blobs plus the industry-standard EICAR
test string, which every antivirus engine is designed to flag WITHOUT
being real malicious code — it exists specifically for safely testing
detection pipelines like this one.

Usage:
    python hash_analyzer.py                 # runs the full demo offline
    VT_API_KEY=xxxx python hash_analyzer.py  # also queries VirusTotal
"""

import os
import csv
import hashlib
import math
import time
from dataclasses import dataclass, field


# ---------------------------------------------------------------------
# 1. Hash generation
# ---------------------------------------------------------------------

def hash_bytes(data: bytes) -> dict:
    """Return MD5, SHA-1, SHA-256 hex digests for a blob of bytes."""
    return {
        "md5": hashlib.md5(data).hexdigest(),
        "sha1": hashlib.sha1(data).hexdigest(),
        "sha256": hashlib.sha256(data).hexdigest(),
    }


def hash_file(path: str, chunk_size: int = 65536) -> dict:
    """Stream a file through all three hash algorithms at once (memory-safe
    for large files — never loads the whole file into RAM)."""
    md5, sha1, sha256 = hashlib.md5(), hashlib.sha1(), hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            md5.update(chunk)
            sha1.update(chunk)
            sha256.update(chunk)
    return {
        "path": path,
        "size_bytes": os.path.getsize(path),
        "md5": md5.hexdigest(),
        "sha1": sha1.hexdigest(),
        "sha256": sha256.hexdigest(),
    }


# ---------------------------------------------------------------------
# 2. VirusTotal lookup (public API v3, hash lookup only — no uploads)
# ---------------------------------------------------------------------

VT_API_KEY = os.environ.get("VT_API_KEY", "")
VT_URL = "https://www.virustotal.com/api/v3/files/{hash}"


def query_virustotal(file_hash: str) -> dict:
    """
    Look up a hash on VirusTotal. Requires a free API key
    (https://www.virustotal.com/gui/join-us) set as the VT_API_KEY
    environment variable. Only sends the hash — never uploads file
    content — so it's safe to run against real suspect files too.

    Returns a dict with detection stats, or {"status": "not_found"} /
    {"status": "no_api_key"} / {"status": "error", ...} as appropriate.
    """
    if not VT_API_KEY:
        return {"status": "no_api_key"}

    try:
        import requests
    except ImportError:
        return {"status": "error", "message": "install the 'requests' package"}

    headers = {"x-apikey": VT_API_KEY}
    try:
        resp = requests.get(VT_URL.format(hash=file_hash), headers=headers, timeout=15)
    except Exception as e:
        return {"status": "error", "message": str(e)}

    if resp.status_code == 404:
        return {"status": "not_found"}
    if resp.status_code == 429:
        return {"status": "error", "message": "rate limited — VT free tier allows 4 req/min"}
    if resp.status_code != 200:
        return {"status": "error", "message": f"HTTP {resp.status_code}"}

    data = resp.json().get("data", {}).get("attributes", {})
    stats = data.get("last_analysis_stats", {})
    return {
        "status": "found",
        "malicious": stats.get("malicious", 0),
        "suspicious": stats.get("suspicious", 0),
        "undetected": stats.get("undetected", 0),
        "total_engines": sum(stats.values()) if stats else 0,
        "type_description": data.get("type_description", "unknown"),
        "names": data.get("names", [])[:5],
    }


def batch_query_virustotal(hashes: list, delay_seconds: float = 15.0) -> dict:
    """
    Query multiple hashes, respecting VT's free-tier rate limit
    (4 requests/minute -> ~15s between calls). Skips politely if no
    API key is configured.
    """
    results = {}
    if not VT_API_KEY:
        print("  [!] No VT_API_KEY set — skipping live VirusTotal queries.")
        print("      Get a free key at https://www.virustotal.com/gui/join-us")
        return {h: {"status": "no_api_key"} for h in hashes}

    for i, h in enumerate(hashes):
        results[h] = query_virustotal(h)
        if i < len(hashes) - 1:
            time.sleep(delay_seconds)
    return results


# ---------------------------------------------------------------------
# 3. Local hash-based detection database
# ---------------------------------------------------------------------

# Known-malicious reference hashes that are PUBLIC and SAFE to store —
# the EICAR test file is the industry-standard AV test string, not real
# malware. In a real lab you'd import full hash sets from NSRL/MalwareBazaar.
EICAR_STRING = rb'X5O!P%@AP[4\PZX54(P^)7CC)7}$EICAR-STANDARD-ANTIVIRUS-TEST-FILE!$H+H*'
EICAR_HASHES = hash_bytes(EICAR_STRING)  # sha256: 275a021b...

KNOWN_MALICIOUS_DB = {
    EICAR_HASHES["sha256"]: {
        "family": "EICAR-Test-File",
        "severity": "test-signature",
        "source": "eicar.org (safe AV test string, not real malware)",
    },
}


@dataclass
class DetectionResult:
    path: str
    md5: str
    sha1: str
    sha256: str
    local_match: dict = None
    vt_result: dict = None
    verdict: str = "unknown"


def build_hash_database(sample_hashes: list, csv_path: str = "malware_hash_db.csv"):
    """Write a CSV database of file hashes — the deliverable for this experiment."""
    fieldnames = ["path", "size_bytes", "md5", "sha1", "sha256", "local_match", "vt_malicious", "vt_total_engines"]
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in sample_hashes:
            writer.writerow({k: row.get(k, "") for k in fieldnames})
    return csv_path


def detect(file_info: dict, use_vt: bool = False) -> DetectionResult:
    result = DetectionResult(
        path=file_info["path"], md5=file_info["md5"],
        sha1=file_info["sha1"], sha256=file_info["sha256"],
    )

    if file_info["sha256"] in KNOWN_MALICIOUS_DB:
        result.local_match = KNOWN_MALICIOUS_DB[file_info["sha256"]]
        result.verdict = "malicious (local DB match)"
    else:
        result.verdict = "clean (no local match)"

    if use_vt:
        vt = query_virustotal(file_info["sha256"])
        result.vt_result = vt
        if vt.get("status") == "found" and vt.get("malicious", 0) > 0:
            result.verdict = f"malicious ({vt['malicious']}/{vt['total_engines']} VT engines)"

    return result


# ---------------------------------------------------------------------
# 4. Hash collision risk analysis (math only — no collision generation)
# ---------------------------------------------------------------------

def birthday_bound_collision_probability(hash_bits: int, num_hashes: int) -> float:
    """
    Approximate probability of at least one collision among `num_hashes`
    random hash outputs of `hash_bits` length, via the birthday-bound
    approximation: P ~= 1 - e^(-n^2 / (2 * 2^bits))
    """
    space = 2 ** hash_bits
    exponent = -(num_hashes ** 2) / (2 * space)
    return 1 - math.exp(exponent)


def hashes_needed_for_probability(hash_bits: int, target_prob: float = 0.5) -> float:
    """Roughly how many random hashes you'd need to generate before
    hitting `target_prob` chance of a collision (birthday bound)."""
    space = 2 ** hash_bits
    return math.sqrt(2 * space * -math.log(1 - target_prob))


def collision_risk_report():
    print("Hash Collision Risk Analysis (birthday-bound approximation)")
    print("-" * 65)
    algos = [("MD5", 128), ("SHA-1", 160), ("SHA-256", 256)]
    for name, bits in algos:
        n50 = hashes_needed_for_probability(bits, 0.5)
        print(f"{name:8s} ({bits:3d}-bit): ~{n50:.3e} random hashes needed for 50% collision odds")
    print()
    print("Practical notes:")
    print("  - MD5 is cryptographically BROKEN: real (non-birthday-bound)")
    print("    collision attacks exist and run in seconds on commodity hardware.")
    print("    Malware authors have historically abused this to make a malicious")
    print("    file share an MD5 with a benign one — never trust MD5 alone.")
    print("  - SHA-1 has practical collision attacks too (e.g. 'SHAttered', 2017).")
    print("  - SHA-256 has no known practical collision attack; it's the right")
    print("    choice for malware identification and should be your primary key.")
    print("  - Best practice: use SHA-256 as the primary identifier, and treat")
    print("    MD5/SHA-1 as legacy fields for cross-referencing older databases")
    print("    (e.g. VirusTotal, NSRL) that still index by them.")
    print()


# ---------------------------------------------------------------------
# 5. Hash-family clustering via simplified fuzzy hashing
# ---------------------------------------------------------------------
# Cryptographic hashes (MD5/SHA*) are deliberately decorrelated — a 1-byte
# change scrambles the whole digest, so you CANNOT cluster "similar" files
# by comparing SHA-256 values. Real tools use fuzzy/similarity hashing
# (ssdeep, TLSH). Below is a small educational context-triggered piecewise
# hash (CTPH) — a simplified stand-in for ssdeep — good enough to
# demonstrate the clustering concept on the sample set.

def simple_fuzzy_hash(data: bytes, window: int = 4, block_divisor: int = 32) -> str:
    """A minimal CTPH-style rolling hash: emits a base64-ish signature that
    stays similar for files that are mostly-similar. Not cryptographically
    rigorous — for teaching the clustering concept only."""
    if len(data) < window:
        return hashlib.md5(data).hexdigest()[:8]

    block_size = max(3, len(data) // block_divisor)
    signature = []
    roll = 0
    for i in range(len(data) - window + 1):
        roll = sum(data[i:i + window]) % 256
        if roll % block_size == 0:
            signature.append(data[i] % 64)
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/"
    return "".join(alphabet[b % len(alphabet)] for b in signature) or "0"


def fuzzy_similarity(sig_a: str, sig_b: str) -> float:
    """Rough similarity score (0-100) between two fuzzy-hash signatures,
    based on longest common substring length relative to signature size."""
    if not sig_a or not sig_b:
        return 0.0
    longer, shorter = (sig_a, sig_b) if len(sig_a) >= len(sig_b) else (sig_b, sig_a)
    best = 0
    for i in range(len(shorter)):
        for j in range(i + 1, len(shorter) + 1):
            if shorter[i:j] in longer:
                best = max(best, j - i)
    return round(100 * best / max(len(sig_a), len(sig_b)), 1)


def cluster_by_family(samples: list, threshold: float = 40.0) -> list:
    """Greedy clustering: group samples whose fuzzy-hash similarity exceeds
    `threshold` into the same family cluster."""
    clusters = []
    for s in samples:
        placed = False
        for cluster in clusters:
            rep = cluster[0]
            sim = fuzzy_similarity(s["fuzzy"], rep["fuzzy"])
            if sim >= threshold:
                cluster.append(s)
                placed = True
                break
        if not placed:
            clusters.append([s])
    return clusters


# ---------------------------------------------------------------------
# 6. Demo driver — synthetic sample "files" (safe, no real malware)
# ---------------------------------------------------------------------

def build_demo_samples() -> list:
    """
    Create a handful of harmless in-memory sample blobs that simulate a
    malware family: a 'base' variant and two near-identical mutations
    (as real polymorphic malware families often look byte-wise), plus
    one unrelated file and the EICAR test string.
    """
    base = b"MZ" + os.urandom(400) + b"SECTION.text" + b"\x90" * 50 + b"malicious_stub_v1"
    variant_a = base[:200] + b"\x00\x00" + base[202:]           # 2-byte patch
    variant_b = base[:350] + os.urandom(20) + base[370:]         # small payload swap
    unrelated = os.urandom(500)

    samples_raw = {
        "trojan_base.bin": base,
        "trojan_variant_a.bin": variant_a,
        "trojan_variant_b.bin": variant_b,
        "unrelated_file.bin": unrelated,
        "eicar_test.com": EICAR_STRING,
    }

    samples = []
    for name, data in samples_raw.items():
        h = hash_bytes(data)
        samples.append({
            "path": name,
            "size_bytes": len(data),
            "md5": h["md5"],
            "sha1": h["sha1"],
            "sha256": h["sha256"],
            "fuzzy": simple_fuzzy_hash(data),
        })
    return samples


def main():
    print("=" * 70)
    print("STEP 1: Generating hashes for sample files")
    print("=" * 70)
    samples = build_demo_samples()
    for s in samples:
        print(f"{s['path']:24s} size={s['size_bytes']:5d}  sha256={s['sha256'][:16]}...")

    print("\n" + "=" * 70)
    print("STEP 2: Local + VirusTotal detection")
    print("=" * 70)
    detections = []
    for s in samples:
        result = detect(s, use_vt=bool(VT_API_KEY))
        detections.append(result)
        vt_note = ""
        if result.vt_result and result.vt_result.get("status") == "found":
            vt_note = f" | VT: {result.vt_result['malicious']}/{result.vt_result['total_engines']} flagged"
        elif result.vt_result and result.vt_result.get("status") == "no_api_key":
            vt_note = " | VT: skipped (no API key)"
        print(f"{s['path']:24s} -> {result.verdict}{vt_note}")
        s["local_match"] = result.local_match["family"] if result.local_match else ""
        s["vt_malicious"] = result.vt_result.get("malicious", "") if result.vt_result else ""
        s["vt_total_engines"] = result.vt_result.get("total_engines", "") if result.vt_result else ""

    print("\n" + "=" * 70)
    print("STEP 3: Writing hash database CSV")
    print("=" * 70)
    csv_path = build_hash_database(samples, "malware_hash_db.csv")
    print(f"Wrote {len(samples)} rows -> {csv_path}")

    print("\n" + "=" * 70)
    print("STEP 4: Collision risk analysis")
    print("=" * 70)
    collision_risk_report()

    print("=" * 70)
    print("STEP 5: Hash-family clustering (fuzzy hash)")
    print("=" * 70)
    clusters = cluster_by_family(samples)
    for i, cluster in enumerate(clusters, 1):
        names = ", ".join(c["path"] for c in cluster)
        print(f"Cluster {i} ({len(cluster)} sample(s)): {names}")

    print("\n" + "=" * 70)
    print("DETECTION RATE STATISTICS")
    print("=" * 70)
    total = len(detections)
    flagged_local = sum(1 for d in detections if d.local_match)
    flagged_vt = sum(1 for d in detections if d.vt_result and d.vt_result.get("status") == "found" and d.vt_result.get("malicious", 0) > 0)
    print(f"Total samples:              {total}")
    print(f"Flagged by local hash DB:   {flagged_local} ({100*flagged_local/total:.0f}%)")
    if VT_API_KEY:
        print(f"Flagged by VirusTotal:      {flagged_vt} ({100*flagged_vt/total:.0f}%)")
    else:
        print(f"Flagged by VirusTotal:      N/A (set VT_API_KEY env var to enable)")
    print(f"Hash-family clusters found: {len(clusters)}")


if __name__ == "__main__":
    main()

STEP 1: Generating hashes for sample files
trojan_base.bin          size=  481  sha256=e1d52fb0814a2748...
trojan_variant_a.bin     size=  481  sha256=bc92a1c1468d4375...
trojan_variant_b.bin     size=  481  sha256=7dfdb5b013e7dbe9...
unrelated_file.bin       size=  500  sha256=7f2195a9bd39ad8d...
eicar_test.com           size=   68  sha256=275a021bbfb6489e...

STEP 2: Local + VirusTotal detection
trojan_base.bin          -> clean (no local match)
trojan_variant_a.bin     -> clean (no local match)
trojan_variant_b.bin     -> clean (no local match)
unrelated_file.bin       -> clean (no local match)
eicar_test.com           -> malicious (local DB match)

STEP 3: Writing hash database CSV
Wrote 5 rows -> malware_hash_db.csv

STEP 4: Collision risk analysis
Hash Collision Risk Analysis (birthday-bound approximation)
-----------------------------------------------------------------
MD5      (128-bit): ~2.172e+19 random hashes needed for 50% collision odds
SHA-1    (160-bit): ~1.423e+24 rand